In [1]:
#Load Dataset
import pandas as pd
df = pd.read_csv('/kaggle/input/heart-disease-dataset/heart.csv')
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


This is a Supervised binary classification dataset

**Handling Duplicates**

In [2]:
df.duplicated().sum()

np.int64(723)

In [3]:
df.drop_duplicates(inplace=True)

In [4]:
df.duplicated().sum()

np.int64(0)

**Handling Missing Values**

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 302 entries, 0 to 878
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       302 non-null    int64  
 1   sex       302 non-null    int64  
 2   cp        302 non-null    int64  
 3   trestbps  302 non-null    int64  
 4   chol      302 non-null    int64  
 5   fbs       302 non-null    int64  
 6   restecg   302 non-null    int64  
 7   thalach   302 non-null    int64  
 8   exang     302 non-null    int64  
 9   oldpeak   302 non-null    float64
 10  slope     302 non-null    int64  
 11  ca        302 non-null    int64  
 12  thal      302 non-null    int64  
 13  target    302 non-null    int64  
dtypes: float64(1), int64(13)
memory usage: 35.4 KB


No missing values

In [6]:
#Imports
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

In [7]:
X=df.drop('target',axis=1)
y=df['target']

In [8]:
# Identify numerical and categorical columns
cat_cols = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
num_cols = [col for col in X.columns if col not in cat_cols]
print(num_cols)

['age', 'trestbps', 'chol', 'thalach', 'oldpeak']


In [9]:
# Preprocessing pipelines

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

#train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [10]:
y.value_counts()
maj_cls=y.value_counts().max()
min_cls=y.value_counts().min()
ratio = maj_cls/min_cls
print(f'class imbalance ratio:{ratio:0.2f}')

class imbalance ratio:1.19


In [11]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.naive_bayes import GaussianNB

pipe_lr = Pipeline([
    ('prep', preprocess),
    ('smote', SMOTE(random_state=42)),
    ('fs', SelectKBest(k=8)),
    ('model', LogisticRegression(max_iter=1000))
])

pipe_dt = Pipeline([
    ('prep', preprocess),
    ('smote', SMOTE(random_state=42)),
    ('model', DecisionTreeClassifier())
])

pipe_rf = Pipeline([
    ('prep', preprocess),
    ('smote', SMOTE(random_state=42)),
    ('model', RandomForestClassifier())
])

pipe_nb = Pipeline([
    ('prep', preprocess),
    ('smote', SMOTE(random_state=42)),
    ('model', GaussianNB())
])

pipe_svc = Pipeline([
    ('prep', preprocess),
    ('smote', SMOTE(random_state=42)),
    ('model', SVC(probability=True))
])

In [12]:
pipe_lr.fit(X_train, y_train)
pipe_dt.fit(X_train, y_train)
pipe_rf.fit(X_train, y_train)
pipe_nb.fit(X_train, y_train)
pipe_svc.fit(X_train, y_train)

acc_lr  = accuracy_score(y_train, pipe_lr.predict(X_train))
acc_dt  = accuracy_score(y_train, pipe_dt.predict(X_train))
acc_rf  = accuracy_score(y_train, pipe_rf.predict(X_train))
acc_nb  = accuracy_score(y_train, pipe_nb.predict(X_train))
acc_svc = accuracy_score(y_train, pipe_svc.predict(X_train))

acc_table = pd.DataFrame({
    "Model": ["Logistic","Decision Tree","Random Forest","Naive Bayes","SVC"],
    "Accuracy": [acc_lr, acc_dt, acc_rf, acc_nb, acc_svc]
})

acc_table.sort_values("Accuracy", ascending=False)

,Model,Accuracy
1,Decision Tree,1.000000
2,Random Forest,1.000000
4,SVC,0.929461
0,Logistic,0.854772
3,Naive Bayes,0.854772


In [13]:
pipelines = {
    'Logistic Regression': pipe_lr,
    'Decision Tree': pipe_dt,
    'Random Forest': pipe_rf,
    'Naive Bayes': pipe_nb,
    'SVM': pipe_svc
}


In [14]:
results = []

for name, pipe in pipelines.items():
    # Train
    pipe.fit(X_train, y_train)

    # Predictions
    y_train_pred = pipe.predict(X_train)
    y_test_pred = pipe.predict(X_test)

    # Metrics
    results.append({
        'Model': name,
        'Train Accuracy': accuracy_score(y_train, y_train_pred),
        'Test Accuracy': accuracy_score(y_test, y_test_pred),

    })

results_df = pd.DataFrame(results)
results_df

,Model,Train Accuracy,Test Accuracy
0,Logistic Regression,0.854772,0.836066
1,Decision Tree,1.000000,0.721311
2,Random Forest,1.000000,0.803279
3,Naive Bayes,0.854772,0.803279
4,SVM,0.929461,0.786885


Logistic regression is the best model here

In [15]:
import joblib

joblib.dump(pipe_lr, "best_model.pkl")

['best_model.pkl']